In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 14. 어텐션 메커니즘의 수학 — 쿼리·키·값 ⭐

> **학습 목표**
> - Q, K, V의 의미를 정보검색(database lookup) 관점으로 설명한다
> - Scaled Dot-Product Attention의 $\sqrt{d_k}$로 나누는 이유를 유도한다
> - 어텐션의 시간/공간 복잡도 $O(n^2 d)$를 분석한다
> - 어텐션 연산을 CPU vs GPU에서 직접 비교한다

## 14.1 어텐션의 등장 배경

RNN의 병목: 인코더의 모든 정보가 마지막 은닉 상태 하나로 압축. 긴 문장에서 정보 손실.

Bahdanau 어텐션(2014): 디코더가 매 스텝마다 인코더의 **모든** 은닉 상태를 참조.

**Self-Attention** (트랜스포머): 같은 시퀀스 내 모든 토큰이 서로를 참조. RNN 없이도 전역 의존성 학습.

## 14.2 쿼리·키·값 (Q, K, V) — 정보검색 관점

데이터베이스 검색에 비유:
- **Query (Q)**: 내가 찾고 싶은 것
- **Key (K)**: 각 항목의 색인 (이 항목이 무엇에 관한 것인지)
- **Value (V)**: 각 항목의 실제 내용

어텐션: Q와 모든 K의 유사도를 계산 → 유사도로 가중합하여 V 반환.

수식:
$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

- $Q \in \mathbb{R}^{n \times d_k}$: $n$개 쿼리
- $K \in \mathbb{R}^{n \times d_k}$: $n$개 키
- $V \in \mathbb{R}^{n \times d_v}$: $n$개 값
- $QK^\top \in \mathbb{R}^{n \times n}$: 각 쿼리-키 쌍의 점수


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Scaled Dot-Product Attention 구현 (NumPy)
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    """Scaled Dot-Product Attention.
    Q, K, V: (n, d) or (batch, n, d)
    """
    d_k = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)  # (..., n, n)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores, axis=-1)
    output = weights @ V
    return output, weights

# 작은 예시
np.random.seed(0)
n, d = 4, 8
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)

output, weights = attention(Q, K, V)
print(f"Q shape: {Q.shape}")
print(f"Attention output: {output.shape}")
print(f"Attention weights: {weights.shape}")
print(f"\n각 행의 Sum (softmax 확인): {weights.sum(axis=-1).round(4)}")


## 14.3 왜 $\sqrt{d_k}$로 나누는가?

$QK^\top$의 원소는 $\sum_{i=1}^{d_k} Q_i K_i$. Q, K가 $\mathcal{N}(0, 1)$이면:

$$\mathrm{Var}(QK^\top_{ij}) = \sum_{i=1}^{d_k} \mathrm{Var}(Q_i K_i) = d_k$$

즉 점수의 표준편차 = $\sqrt{d_k}$. $d_k$가 크면 점수가 커져 softmax가 **포화** (한 값만 1, 나머지 0).

$\sqrt{d_k}$로 나누면 분산이 1로 정규화 → softmax가 부드러운 분포 유지. 그래디언트도 잘 흐름.


In [ ]:
# sqrt(d_k) 스케일링 효과 실험
np.random.seed(0)
dks = [8, 64, 512, 2048]
fig, axes = plt.subplots(1, len(dks), figsize=(16, 4))

for ax, d_k in zip(axes, dks):
    n = 4
    Q = np.random.randn(n, d_k)
    K = np.random.randn(n, d_k)
    # 스케일링 없이
    scores_no = Q @ K.T
    # 스케일링 있이
    scores_scaled = scores_no / np.sqrt(d_k)

    # softmax
    p_no = softmax(scores_no)
    p_scaled = softmax(scores_scaled)

    ax.bar(['q0', 'q1', 'q2', 'q3'], p_no[0], alpha=0.5, label='스케일링 X', color='red')
    ax.bar(['q0', 'q1', 'q2', 'q3'], p_scaled[0], alpha=0.5, label='스케일링 O', color='blue')
    ax.set_title(f'd_k={d_k}\nVariance: {scores_no.var():.1f} → {scores_scaled.var():.1f}')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../figures/ch14_scaling_effect.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> d_k가 클수록 스케일링 없으면 softmax가 한 값에 몰린다 (포화).")


## 14.4 어텐션의 시간·공간 복잡도

$$\mathrm{Attn}(Q, K, V) = \mathrm{softmax}(QK^\top / \sqrt{d_k}) V$$

- $QK^\top$: $n \times d \times n = O(n^2 d)$ 연산
- $\mathrm{softmax}(A) V$: $n \times n \times d = O(n^2 d)$
- **총 시간**: $O(n^2 d)$
- **공간**: $A \in \mathbb{R}^{n \times n}$ → $O(n^2)$

문제: 시퀀스 길이 $n$이 길어지면 $n^2$이 폭발. 8K 컨텍스트면 $n^2 = 6.4 \times 10^7$.
→ Flash Attention, 희소 어텐션 등의 해결책 (Ch 27에서).


In [ ]:
# 시퀀스 길이별 연산량 시각화
seq_lens = [128, 256, 512, 1024, 2048, 4096, 8192]
n2 = [n**2 for n in seq_lens]
n2d = [n**2 * 64 for n in seq_lens]  # d=64

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(seq_lens)), n2, alpha=0.7, label='$n^2$ (Space)')
ax.bar(range(len(seq_lens)), [n/1000 for n in n2d], alpha=0.7, label='$n^2 \\cdot d$ / 1000 (Operation)')
ax.set_xticks(range(len(seq_lens)))
ax.set_xticklabels([str(n) for n in seq_lens])
ax.set_xlabel('시퀀스 길이 n')
ax.set_ylabel('Complexity')
ax.set_title('Attention Complexity: $O(n^2 d)$ — 시퀀스 길이에 따라 폭발')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/ch14_complexity.png', dpi=100, bbox_inches='tight')
plt.show()
print("8K Context는 64M 개의 Attention Scores. Flash Attention이 필수인 이유.")


## 14.5 인과적 마스킹 (Causal Masking)

GPT 같은 autoregressive 모델은 미래 토큰을 보면 안 된다.

마스크 행렬 $M$:
$$M_{ij} = \begin{cases} 0 & \text{if } i \geq j \text{ (과거/현재)} \\ -\infty & \text{if } i < j \text{ (미래)} \end{cases}$$

$A + M$에 softmax → 미래 위치의 가중치가 0이 됨.


In [ ]:
# 인과적 마스킹
n = 6
mask = np.triu(np.ones((n, n)) * -1e9, k=1)  # 상단 삼각 -inf
print("Causal Mask (상단 삼각 -inf):")
print(mask[:4, :4])

# 적용 예시
np.random.seed(0)
scores = np.random.randn(n, n)  # 어텐션 점수
print(f"\n원본 Scores (첫 행): {scores[0].round(2)}")
masked = scores + mask
print(f"Mask Application (첫 행): {masked[0].round(2)}")
weights = softmax(masked)
print(f"\n소프트맥스 후 (첫 행): {weights[0].round(4)}")
print(f"  → 첫 번째 토큰은 자기 자신만 본다 (1.0)")

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].imshow(scores, cmap='viridis'); plt.colorbar(im0, ax=axes[0])
axes[0].set_title('원본 Attention Scores')
im1 = axes[1].imshow(mask != -1e9, cmap='binary'); plt.colorbar(im1, ax=axes[1])
axes[1].set_title('Causal Mask (흰=허용)')
im2 = axes[2].imshow(weights, cmap='Blues'); plt.colorbar(im2, ax=axes[2])
axes[2].set_title('Mask Application 후 Weight')
plt.tight_layout()
plt.savefig('../figures/ch14_causal_mask.png', dpi=100, bbox_inches='tight')
plt.show()


## 14.6 [CPU/GPU 벤치마크 ⑤] 시퀀스 길이별 어텐션 시간

시퀀스 길이 $n$이 2배 커지면 어텐션 시간은 4배 (메모리는 4배).
CPU vs GPU 차이를 직접 체감해 보자.


In [ ]:
# 어텐션 벤치마크
import time
from llm_math.bench import time_fn, format_results_table

def attention_torch(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-1, -2) / (d_k ** 0.5)
    if mask is not None:
        scores = scores + mask
    weights = F.softmax(scores, dim=-1)
    return weights @ V

# 시퀀스 길이별
print(f"{'n':>6} | {'CPU (ms)':>12} | {'GPU (ms)':>12} | {'Speedup':>10}")
print("-" * 50)
for n in [128, 256, 512, 1024, 2048]:
    d = 64
    Q = torch.randn(n, d)
    K = torch.randn(n, d)
    V = torch.randn(n, d)
    res_cpu = time_fn(attention_torch, Q, K, V, device='cpu', warmup=2, repeat=3)
    if torch.cuda.is_available():
        Q_g, K_g, V_g = Q.cuda(), K.cuda(), V.cuda()
        res_gpu = time_fn(attention_torch, Q_g, K_g, V_g, device='cuda', warmup=3, repeat=5)
        speedup = res_cpu['mean_ms'] / res_gpu['mean_ms']
        print(f"{n:>6} | {res_cpu['mean_ms']:>12.3f} | {res_gpu['mean_ms']:>12.3f} | {speedup:>9.2f}x")
    else:
        print(f"{n:>6} | {res_cpu['mean_ms']:>12.3f} | {'N/A':>12} | {'N/A':>10}")

if not torch.cuda.is_available():
    print("\n⚠ GPU 없음. n=2048 정도면 CPU에서 눈에 띄게 느려진다.")


In [ ]:
# PyTorch SDPA (scaled_dot_product_attention) 활용
print("PyTorch SDPA Function 사용:")
print("(내부적으로 Flash Attention 등 최적화 백엔드 자동 선택)\n")

n, d = 512, 64
Q = torch.randn(1, n, d)
K = torch.randn(1, n, d)
V = torch.randn(1, n, d)

# 직접 구현
out_manual = attention_torch(Q, K, V)

# PyTorch SDPA (F.scaled_dot_product_attention)
out_sdpa = F.scaled_dot_product_attention(Q, K, V)

print(f"직접 구현 vs SDPA 최대 오차: {(out_manual - out_sdpa).abs().max():.2e}")
print("\n=> 동일한 결과. SDPA는 내부적으로 최적화되어 빠르다.")

# 속도 비교
print("\nSpeed Comparison (n=512):")
t_manual = time_fn(attention_torch, Q, K, V, device='cpu', warmup=2, repeat=5)
def sdpa_call(Q, K, V):
    return F.scaled_dot_product_attention(Q, K, V)
t_sdpa = time_fn(sdpa_call, Q, K, V, device='cpu', warmup=2, repeat=5)
print(f"  직접 구현: {t_manual['mean_ms']:.3f} ms")
print(f"  PyTorch SDPA: {t_sdpa['mean_ms']:.3f} ms")
print(f"  Speed Improvement: {t_manual['mean_ms'] / t_sdpa['mean_ms']:.2f}x")


## 14.7 어텐션 역전파 (개념)

소프트맥스의 야코비안을 포함한 복잡한 미분. 핵심:

$$\frac{\partial \mathcal{L}}{\partial Q} = \frac{1}{\sqrt{d_k}} \left(\frac{\partial \mathcal{L}}{\partial A} V K^\top + \ldots\right)$$

PyTorch의 `autograd`가 알아서 처리. Flash Attention은 이 역전파를 메모리 효율적으로 (Ch 27).

## 14.8 핵심 정리

| 개념 | 수식 | 의미 |
|---|---|---|
| 어텐션 | $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$ | 쿼리-키 유사도로 값 가중합 |
| 스케일링 | $\frac{1}{\sqrt{d_k}}$ | 분산 1로 정규화 |
| 인과적 마스크 | $M_{ij} = -\infty$ if $i < j$ | 미래 참조 차단 |
| 복잡도 | $O(n^2 d)$ 시간, $O(n^2)$ 공간 | 시퀀스 길이에 2차 |

## 연습문제

1. $\mathrm{Var}(QK^\top_{ij}) = d_k$임을 Q, K의 원소가 iid $\mathcal{N}(0,1)$일 때 유도하라.
2. 인과적 마스크가 없는 어텐션과 있는 어텐션의 출력을 비교하라.
3. 어텐션 가중치 행렬을 시각화하고, 대각 성분이 큰 이유를 설명하라.
4. 시퀀스 길이 256, 512, 1024, 2048에서 CPU 시간을 측정하고 $O(n^2)$를 검증하라.
5. PyTorch SDPA의 `is_causal=True` 옵션을 사용해 인과적 어텐션을 구현하라.

> 해설: `solutions/ch14_solutions.ipynb`
